# ✍️ Module 5.2: Model Signatures

**Goal**: Create, infer, and enforce model signatures for input/output validation.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
mlflow.set_experiment("05_model_signatures")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name="quality")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Infer Signature Automatically

The easiest way: let MLFlow infer the signature from your training data and predictions.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Infer the signature
signature = infer_signature(X_train, predictions)

print("📝 Inferred Signature:")
print(f"\nInputs:\n{signature.inputs}")
print(f"\nOutputs:\n{signature.outputs}")

In [ ]:
# Log model with signature
with mlflow.start_run(run_name="model_with_signature"):
    mlflow.log_metric("accuracy", accuracy_score(y_test, predictions))
    
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        signature=signature,         # ← The signature!
        input_example=X_train[:3],   # ← Sample input (shown in UI)
    )
    
    run_id = mlflow.active_run().info.run_id
    print(f"✅ Model logged with signature!")
    print(f"   Run ID: {run_id}")
    print(f"   Check the MLFlow UI → this run → Artifacts → model → MLmodel")

## 2. Signature Enforcement

When loading a model with a signature, MLFlow validates inputs automatically.

In [ ]:
# Load the model
loaded_model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")

# Valid input — works fine
valid_result = loaded_model.predict(X_test[:3])
print(f"Valid input prediction: {valid_result}")

# The signature ensures your model gets the right data format!

In [ ]:
# Invalid input — signature catches the error
try:
    bad_input = pd.DataFrame({"wrong_column": [1, 2, 3]})
    loaded_model.predict(bad_input)
except Exception as e:
    print(f"❌ Signature caught bad input!")
    print(f"   Error: {type(e).__name__}: {str(e)[:200]}")

## 3. Including Params in Signature

You can add **inference parameters** to signatures — useful for controlling prediction behavior.

In [ ]:
from mlflow.models import infer_signature
from mlflow.types.schema import ParamSchema, ParamSpec

# Create signature with params
params = [
    ParamSpec("predict_proba", "boolean", False),
    ParamSpec("top_k", "integer", 1),
]

signature_with_params = infer_signature(
    X_train, 
    predictions,
    params=params
)

print("📝 Signature with inference params:")
print(f"Params: {signature_with_params.params}")

## 🔑 Key Takeaways

1. **`infer_signature(X, predictions)`** — easiest way to create a signature
2. **Always include a signature** when logging models — it's free validation
3. **`input_example`** provides a sample input, visible in the UI
4. Signatures support **column-based** (tabular) and **tensor-based** schemas
5. **Signature enforcement** catches bad inputs at prediction time

---

## ➡️ Next: `03_custom_pyfunc_models.ipynb` — Build custom prediction logic